In [0]:
dbutils.widgets.text("source_system", "", "Source System")
dbutils.widgets.text("source_path_or_table", "", "Source Path or Table")
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")

source_system = dbutils.widgets.get("source_system")
source_path_or_table = dbutils.widgets.get("source_path_or_table")
landing_timestamp = dbutils.widgets.get("landing_timestamp")

In [0]:
landing_path = f"/Volumes/amperedrive_giga/landingadg/landingadg_volume/{source_system}/{source_path_or_table}/{landing_timestamp}/"

# Check whether the landing path (and its parent directories) exists; exit gracefully if not
try:
    dbutils.fs.ls(landing_path)
except Exception as e:
    if "FileNotFoundException" in str(e) or "does not exist" in str(e) or "NOT_FOUND" in str(e):
        msg = f"No new files are received at: {landing_path}"
        print(msg)
        dbutils.notebook.exit(msg)
    raise

if source_system == "POSTGRES_NEON":
    landing_df = spark.read.format("parquet").load(landing_path)
elif source_system == "ADLS_CSV":
    landing_df = spark.read.format("csv").option("header", True).option("inferSchema", True).load(landing_path)
else:
    raise ValueError(f"Unsupported source_system: {source_system}. Expected 'POSTGRES_NEON' or 'ADLS_CSV'.")

In [0]:
from pyspark.sql.functions import lit, current_timestamp

updated_df = landing_df.withColumn("landing_timestamp", lit(landing_timestamp)) \
                       .withColumn("insert_timestamp", current_timestamp())

In [0]:
updated_df.write.mode("append").saveAsTable(f"amperedrive_giga.bronze.{source_path_or_table}")